In [16]:
# run this cell for sanity check 
import sys
print(sys.executable)

# It should print a path containing venv — that's your confirmation everything is wired up correctly. Let me know what you see!

/Users/karthikeyadevaraj/Desktop/sepsis_detector/venv/bin/python


In [17]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import shap
import warnings
#-------------------------
# karthi made updation to imports here 
import random
#---------------------------
warnings.filterwarnings('ignore')

In [18]:
def load_all_patients(set_paths):
    dfs = []
    for folder in set_paths:
        files = os.listdir(folder)
        print(f"Loading {len(files)} files from {folder}...")
        for fname in files:
            if fname.endswith('.psv'):
                path = os.path.join(folder, fname)
                df = pd.read_csv(path, sep='|')
                df['patient_id'] = fname.replace('.psv', '')
                dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(combined)}")
    print(f"Total patients: {combined['patient_id'].nunique()}")
    return combined

# Point these to your actual folders
SET_A = r'/Users/karthikeyadevaraj/Desktop/physionet_2019_data/training/training_setA'
SET_B = r'/Users/karthikeyadevaraj/Desktop/physionet_2019_data/training/training_setB'

data = load_all_patients([SET_A, SET_B])
data.head()

Loading 20336 files from /Users/karthikeyadevaraj/Desktop/physionet_2019_data/training/training_setA...
Loading 20000 files from /Users/karthikeyadevaraj/Desktop/physionet_2019_data/training/training_setB...

Total rows: 1552210
Total patients: 40336


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,80.0,100.0,36.50,121.00,58.0,41.00,13.5,NaN,1.0,25.0,...,223.0,160.0,77.27,1,0.0,1.0,-69.14,3,0,p014977
1,76.0,100.0,36.25,113.25,61.0,41.50,12.0,NaN,1.0,25.0,...,NaN,NaN,77.27,1,0.0,1.0,-69.14,4,0,p014977
2,80.0,100.0,36.25,132.75,71.5,46.25,12.0,NaN,NaN,NaN,...,NaN,NaN,77.27,1,0.0,1.0,-69.14,5,0,p014977
3,78.0,100.0,36.10,103.50,58.0,43.00,12.0,NaN,-3.0,NaN,...,NaN,NaN,77.27,1,0.0,1.0,-69.14,6,0,p014977
4,74.0,100.0,36.00,128.75,69.5,44.50,12.5,NaN,-3.0,NaN,...,NaN,NaN,77.27,1,0.0,1.0,-69.14,7,0,p014977


In [19]:
label_counts = data['SepsisLabel'].value_counts()
print("SepsisLabel counts:")
print(label_counts)

neg = label_counts[0]
pos = label_counts[1]
scale_pos_weight = neg / pos

print(f"\nNon-sepsis rows: {neg}")
print(f"Sepsis rows:     {pos}")
print(f"Ratio:           {scale_pos_weight:.1f}x")
print(f"\nscale_pos_weight to use in XGBoost: {scale_pos_weight:.2f}")

SepsisLabel counts:
SepsisLabel
0    1524294
1      27916
Name: count, dtype: int64

Non-sepsis rows: 1524294
Sepsis rows:     27916
Ratio:           54.6x

scale_pos_weight to use in XGBoost: 54.60


In [32]:
def preprocess(df):
    df = df.copy()
    # #-------- old code was causing issue -----
    # # ── 1. Forward fill within each patient, then backward fill ──
    # df = df.groupby('patient_id', group_keys=False).apply(
    #     lambda x: x.ffill().bfill()
    # )



    # #-------karthi updated new code ---------------
    # # ── 1. Forward fill within each patient, then backward fill ──
    # Save patient_id first because pandas 2.x drops groupby keys after apply()
    patient_ids = df['patient_id'].copy()

    df = df.groupby('patient_id', group_keys=False).apply(
        lambda x: x.ffill().bfill()
    )

    # Restore patient_id if pandas dropped it (happens in pandas 2.0+)
    if 'patient_id' not in df.columns:
        df['patient_id'] = patient_ids
    # #---------------------------------------------
    
    # ── 2. Fill anything still missing with column median ──
    df = df.fillna(df.median(numeric_only=True))
    
    # ── 3. Safety net ──
    df = df.fillna(0)
    
    # ── 4. Rolling mean (last 6 hours) per patient ──
    vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp']
    for col in vitals:
        df[f'{col}_6h_mean'] = (
            df.groupby('patient_id')[col]
            .transform(lambda x: x.rolling(6, min_periods=1).mean())
        )
        df[f'{col}_3h_mean'] = (
            df.groupby('patient_id')[col]
            .transform(lambda x: x.rolling(3, min_periods=1).mean())
        )
    
    # ── 5. Delta features (change from previous hour) ──
    for col in vitals:
        df[f'{col}_delta'] = df.groupby('patient_id')[col].diff().fillna(0)
    
    # ── 6. Ratio feature (shock index) ──
    df['shock_index'] = df['HR'] / (df['SBP'].replace(0, np.nan).fillna(1))
    
    return df

print("Running preprocessing... (this takes 2-4 minutes)")
data = preprocess(data)
print("Done!")
print(f"Total features now: {len(data.columns)}")

Running preprocessing... (this takes 2-4 minutes)
Done!
Total features now: 61


In [21]:
FEATURES = [
    # Raw vitals
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp',
    # Key labs
    'WBC', 'Creatinine', 'Lactate', 'Glucose',
    'Potassium', 'HCO3', 'pH', 'Platelets',
    # Demographics
    'Age', 'Gender', 'HospAdmTime', 'ICULOS',
    # Engineered features
    'HR_6h_mean', 'Resp_6h_mean', 'Temp_6h_mean',
    'SBP_6h_mean', 'O2Sat_6h_mean',
    'HR_3h_mean', 'Resp_3h_mean',
    'HR_delta', 'Resp_delta', 'Temp_delta', 'SBP_delta',
    'shock_index'
]

X = data[FEATURES]
y = data['SepsisLabel']
groups = data['patient_id']

# Split by patient — NOT by row
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
X_test  = X.iloc[test_idx]
y_test  = y.iloc[test_idx]

print(f"Train rows: {len(X_train)}")
print(f"Test rows:  {len(X_test)}")
print(f"Sepsis in train: {y_train.sum()}")
print(f"Sepsis in test:  {y_test.sum()}")

Train rows: 1241213
Test rows:  310997
Sepsis in train: 22669
Sepsis in test:  5247


In [22]:
print(f"Training with scale_pos_weight = {scale_pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1  # use all CPU cores
)

model.fit(X_train, y_train)
print("Training complete!")

Training with scale_pos_weight = 54.60
Training complete!


In [23]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"AUROC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nTarget: AUROC > 0.85")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.87      0.92    305750
           1       0.07      0.62      0.13      5247

    accuracy                           0.86    310997
   macro avg       0.53      0.74      0.53    310997
weighted avg       0.98      0.86      0.91    310997

AUROC: 0.8262

Target: AUROC > 0.85


In [24]:
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/sepsis_xgb_model.pkl')
json.dump(FEATURES, open('models/feature_columns.json', 'w'))

print("Saved:")
print("  models/sepsis_xgb_model.pkl")
print("  models/feature_columns.json")

Saved:
  models/sepsis_xgb_model.pkl
  models/feature_columns.json


In [25]:
explainer = shap.TreeExplainer(model)

# Test on one row
sample = X_test.iloc[[0]]
shap_vals = explainer.shap_values(sample)

# Top 6 features for that row
shap_series = pd.Series(shap_vals[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("Top 6 contributing features for this prediction:")
print(shap_series.head(6))

Top 6 contributing features for this prediction:
SBP_6h_mean     0.808460
ICULOS         -0.486252
Creatinine      0.461335
MAP             0.394034
Resp_6h_mean   -0.291752
HospAdmTime    -0.290976
dtype: float32


In [26]:
# Cell 10 — Save population medians for API use
import json

medians = data[FEATURES].median().to_dict()
with open('models/population_medians.json', 'w') as f:
    json.dump(medians, f)

print("Saved models/population_medians.json")

Saved models/population_medians.json


In [27]:
import os, random

# Find a patient who actually got sepsis
sepsis_files = []
for folder in [SET_A, SET_B]:
    for fname in os.listdir(folder):
        if fname.endswith('.psv'):
            df_temp = pd.read_csv(os.path.join(folder, fname), sep='|')
            if df_temp['SepsisLabel'].sum() > 0:
                sepsis_files.append(os.path.join(folder, fname))

# Pick a random sepsis patient
test_file = random.choice(sepsis_files[:100])
print(f"Testing on: {test_file}")

# Load and preprocess
patient_df = pd.read_csv(test_file, sep='|')
patient_df['patient_id'] = 'test'
processed = preprocess(patient_df)

# Predict
X_patient = processed[FEATURES]
scores = model.predict_proba(X_patient)[:, 1]

print(f"\nHours in ICU: {len(scores)}")
print(f"Max risk score: {scores.max():.3f}")
print(f"Hour of peak risk: {scores.argmax()}")
print(f"Alert triggered (>0.65): {(scores > 0.65).any()}")
print(f"\nRisk scores by hour:")
for i, s in enumerate(scores):
    bar = '█' * int(s * 20)
    flag = ' ⚠️ ALERT' if s > 0.65 else ''
    print(f"  Hour {i+1:3d}: {s:.3f} {bar}{flag}")

Testing on: /Users/karthikeyadevaraj/Desktop/physionet_2019_data/training/training_setA/p012269.psv

Hours in ICU: 42
Max risk score: 0.299
Hour of peak risk: 41
Alert triggered (>0.65): False

Risk scores by hour:
  Hour   1: 0.115 ██
  Hour   2: 0.115 ██
  Hour   3: 0.149 ██
  Hour   4: 0.173 ███
  Hour   5: 0.154 ███
  Hour   6: 0.138 ██
  Hour   7: 0.140 ██
  Hour   8: 0.145 ██
  Hour   9: 0.132 ██
  Hour  10: 0.130 ██
  Hour  11: 0.207 ████
  Hour  12: 0.194 ███
  Hour  13: 0.192 ███
  Hour  14: 0.204 ████
  Hour  15: 0.200 ███
  Hour  16: 0.291 █████
  Hour  17: 0.291 █████
  Hour  18: 0.247 ████
  Hour  19: 0.247 ████
  Hour  20: 0.204 ████
  Hour  21: 0.242 ████
  Hour  22: 0.149 ██
  Hour  23: 0.144 ██
  Hour  24: 0.145 ██
  Hour  25: 0.148 ██
  Hour  26: 0.282 █████
  Hour  27: 0.271 █████
  Hour  28: 0.276 █████
  Hour  29: 0.179 ███
  Hour  30: 0.264 █████
  Hour  31: 0.287 █████
  Hour  32: 0.220 ████
  Hour  33: 0.257 █████
  Hour  34: 0.234 ████
  Hour  35: 0.292 █████
 

In [28]:
# Find a patient who never got sepsis
healthy_files = []
for folder in [SET_A, SET_B]:
    for fname in os.listdir(folder):
        if fname.endswith('.psv'):
            df_temp = pd.read_csv(os.path.join(folder, fname), sep='|')
            if df_temp['SepsisLabel'].sum() == 0:
                healthy_files.append(os.path.join(folder, fname))

test_file_healthy = random.choice(healthy_files[:100])
print(f"Testing on: {test_file_healthy}")

patient_df2 = pd.read_csv(test_file_healthy, sep='|')
patient_df2['patient_id'] = 'test'
processed2 = preprocess(patient_df2)
X_patient2 = processed2[FEATURES]
scores2 = model.predict_proba(X_patient2)[:, 1]

print(f"\nMax risk score: {scores2.max():.3f}")
print(f"Alert triggered (>0.65): {(scores2 > 0.65).any()}")
print("✅ Good: no false alarm" if not (scores2 > 0.65).any() else "⚠️ False alarm!")

Testing on: /Users/karthikeyadevaraj/Desktop/physionet_2019_data/training/training_setA/p002873.psv

Max risk score: 0.269
Alert triggered (>0.65): False
✅ Good: no false alarm
